In [1]:
import os
import sys
import torch
from torch.utils.tensorboard.writer import SummaryWriter
import logging
import pickle
import cv2
import numpy as np

In [2]:
repo_path = os.path.abspath('/home/ids/vimanach/repos/insightface/recognition')
sys.path.insert(0, repo_path)

In [3]:
from arcface_torch.backbones import get_model
from arcface_torch.eval import verification

In [4]:
@torch.no_grad()
def load_bin(path, image_size):
    try:
        with open(path, 'rb') as f:
            bins, issame_list = pickle.load(f)  # py2
    except UnicodeDecodeError:
        with open(path, 'rb') as f:
            bins, issame_list = pickle.load(f, encoding='bytes')  # py3
    data_list = []
    for flip in [0, 1]:
        data = torch.empty((len(issame_list) * 2, 3, image_size[0], image_size[1]))
        data_list.append(data)
    for idx in range(len(issame_list) * 2):
        _bin = bins[idx]
        img = cv2.imdecode(np.frombuffer(_bin, dtype=np.uint8), cv2.IMREAD_COLOR)
        assert img is not None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if img.shape[1] != image_size[0]:
            h, w = img.shape[:2]
            if h < w:
                newh = image_size[0]
                neww = int(w * (image_size[0] / h))
            else:
                neww = image_size[0]
                newh = int(h * (image_size[0] / w))
            img = cv2.resize(img, (neww, newh))
        img = np.transpose(img, axes=(2, 0, 1))
        for flip in [0, 1]:
            if flip == 1:
                img_flip = np.flip(img, axis=2).copy()
                data_list[flip][idx][:] = torch.from_numpy(img_flip)
            else:
                data_list[flip][idx][:] = torch.from_numpy(img)
        if idx % 1000 == 0:
            print(f'[inside load_bin func] loading bin {idx}')
    print(f"[inside load_bin func] data_list[0].shape={data_list[0].shape}")
    return data_list, issame_list

In [5]:
class CallBackVerification(object):
    
    def __init__(self, val_targets, rec_prefix, summary_writer=None, image_size=(112, 112), wandb_logger=None):
        self.highest_acc: float = 0.0
        self.highest_acc_list: list[float] = [0.0] * len(val_targets)
        self.ver_list: list[object] = []
        self.ver_name_list: list[str] = []
        self.init_dataset(val_targets=val_targets, data_dir=rec_prefix, image_size=image_size)

        self.summary_writer = summary_writer
        self.wandb_logger = wandb_logger

    def ver_test(self, backbone: torch.nn.Module, global_step: int):
        results = []
        for i in range(len(self.ver_list)):
            acc1, std1, acc2, std2, xnorm, embeddings_list = verification.test(
                self.ver_list[i], backbone, 10, 10)
            logging.info('[%s][%d]XNorm: %f' % (self.ver_name_list[i], global_step, xnorm))
            logging.info('[%s][%d]Accuracy-Flip: %1.5f+-%1.5f' % (self.ver_name_list[i], global_step, acc2, std2))

            if acc2 > self.highest_acc_list[i]:
                self.highest_acc_list[i] = acc2
            logging.info(
                '[%s][%d]Accuracy-Highest: %1.5f' % (self.ver_name_list[i], global_step, self.highest_acc_list[i]))
            results.append(acc2)

    def init_dataset(self, val_targets, data_dir, image_size):
        for name in val_targets:
            path = os.path.join(data_dir, name + ".bin")
            if os.path.exists(path):
                data_set = load_bin(path, image_size)
                self.ver_list.append(data_set)
                self.ver_name_list.append(name)
                print(f"[CallBackVerification, init_dataset] add validation dataset `{name}` from `{path}`.")

    def __call__(self, num_update, backbone: torch.nn.Module):
        if num_update > 0:
            backbone.eval()
            self.ver_test(backbone, num_update)
            backbone.train()

In [6]:
val_targets = ['African_test', 'Asian_test', 'Caucasian_test', 'Indian_test']
rec = os.environ["DATA_DIR"] + "/bupt-balancedface-mxnet_biased_African_10pct/flattened__Caucasian_African"

callback_verification = CallBackVerification(
      val_targets=val_targets, rec_prefix=rec, 
      summary_writer=None, wandb_logger = None
  )

[inside load_bin func] loading bin 0
[inside load_bin func] loading bin 1000
[inside load_bin func] loading bin 2000
[inside load_bin func] loading bin 3000
[inside load_bin func] loading bin 4000
[inside load_bin func] loading bin 5000
[inside load_bin func] loading bin 6000
[inside load_bin func] loading bin 7000
[inside load_bin func] loading bin 8000
[inside load_bin func] loading bin 9000
[inside load_bin func] loading bin 10000
[inside load_bin func] loading bin 11000
[inside load_bin func] data_list[0].shape=torch.Size([12000, 3, 112, 112])
[CallBackVerification, init_dataset] add validation dataset `African_test` from `/home/ids/vimanach/datasets/bupt-balancedface-mxnet_biased_African_10pct/flattened__Caucasian_African/African_test.bin`.
[inside load_bin func] loading bin 0
[inside load_bin func] loading bin 1000
[inside load_bin func] loading bin 2000
[inside load_bin func] loading bin 3000
[inside load_bin func] loading bin 4000
[inside load_bin func] loading bin 5000
[inside

In [7]:
callback_verification.ver_name_list

['African_test', 'Asian_test', 'Caucasian_test', 'Indian_test']

In [8]:
backbone = get_model("r50", dropout=0.0, fp16=False, num_features=512).cuda()

In [9]:
ckpt_path = os.environ["ARTIFACT_DIR"] + "/trained/arcface_torch/r50/bupt-balancedface-mxnet_biased_African_10pct/Caucasian_African/model.pt"
checkpoint = torch.load(ckpt_path, map_location='cuda', weights_only=True)
backbone.load_state_dict(checkpoint)

<All keys matched successfully>

In [10]:
callback_verification(1, backbone)

testing verification..
(12000, 512)
infer time 42.86428899999989
testing verification..
(12000, 512)
infer time 42.222881000000044
testing verification..
(12000, 512)
infer time 42.06272699999998
testing verification..
(12000, 512)
infer time 42.04979099999995
